# [AI] ROTBOTS -- Full Pipeline
## Topic -> Video in One Notebook

---

All 9 pipeline stages in one place. Run cells top to bottom.

> **[?]** Every cell is an automated decision. What does the machine choose?

---

In [ ]:
# SETUP (all dependencies)
!pip install -q requests beautifulsoup4 pymupdf edge-tts fal-client yt-dlp faster-whisper
import json, re, random, requests, subprocess, shutil, os, time as _time, threading, edge_tts
from pathlib import Path
from bs4 import BeautifulSoup
from IPython.display import display, Markdown, HTML, Audio, Video

from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
BASE_DIR.mkdir(parents=True, exist_ok=True)
TEMP = Path('/content/temp_assembly'); TEMP.mkdir(exist_ok=True)

# Progress helpers
def progress_bar(c,t,l='',w=30):
    p=c/t if t>0 else 0;f=int(w*p);return f'<div style="font-family:monospace;font-size:14px;padding:2px 0;">{"#"*f}{"-"*(w-f)} {c}/{t} {l} ({p:.0%})</div>'
def progress_html(title,c,t,l='',d='',color='#4ecca3'):
    return f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;"><div style="font-size:16px;font-weight:bold;color:{color};">{title}</div>{progress_bar(c,t,l)}' + (f'<div style="color:#a0a0a0;font-size:12px;margin-top:4px;">{d}</div>' if d else '') + '</div>'

print('[OK] All dependencies ready')


In [ ]:
# API KEYS
GROQ_API_KEY = ''   # Free: https://console.groq.com/keys
FAL_API_KEY = ''    # Video: https://fal.ai/dashboard/keys

if GROQ_API_KEY: print('[OK] Groq')
else: print('[!] Paste GROQ_API_KEY above')
if FAL_API_KEY: os.environ['FAL_KEY']=FAL_API_KEY; print('[OK] fal.ai')
else: print('[!] Paste FAL_API_KEY above')



---

# ==== 01: Video Plan


# [SCENE] ROTBOTS -- Video Plan

---

Configure your video before building it. This plan controls everything downstream.

> **[?]** You're about to design a content machine. What decisions are you making vs. the machine?

---

In [ ]:
# SETUP
import json, re
from pathlib import Path
from IPython.display import display, Markdown
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
BASE_DIR.mkdir(parents=True, exist_ok=True)
print('[OK] Setup')

---
## [PLAN] Video Plan

Set the total length, content mix, and features for your video.

In [ ]:
# ============================================================
# VIDEO PLAN -- Configure your video here
# ============================================================

TOPIC = 'The history and ethics of AI-generated art'

SOURCES = [
    'https://en.wikipedia.org/wiki/Artificial_intelligence_art',
    # --- Add your sources below (websites, PDFs, or raw text) ---
    # 'https://example.com/any-article-or-blog-post',
    # 'https://arxiv.org/pdf/2302.06576',              # PDF link
    # 'https://www.nytimes.com/2023/ai-art-debate',     # news article
    # 'https://medium.com/@someone/my-essay-123',       # blog post
    # 'Raw text also works: just paste your text here as a string.',
]

# Total video length
TOTAL_VIDEO_LENGTH = 60    # seconds (including credits)
SECONDS_PER_SCENE = 5      # each scene/clip length

# Content mix (must add up to <= 1.0, rest = AI-generated)
ARCHIVE_RATIO = 0.0        # 0.0-1.0 -- archive.org footage
UPLOAD_RATIO = 0.0          # 0.0-1.0 -- your own uploaded footage

# Optional features
ENABLE_CREDITS = True
ENABLE_SUBTITLES = False
ENABLE_MUSIC = False
ENABLE_EFFECTS = True

SESSION_NAME = ''  # Leave empty to auto-generate

In [ ]:
# CALCULATE PLAN
AI_RATIO = max(0, 1.0 - ARCHIVE_RATIO - UPLOAD_RATIO)
if AI_RATIO + ARCHIVE_RATIO + UPLOAD_RATIO > 1.0:
    print('[!!] Ratios exceed 1.0!'); AI_RATIO=1.0; ARCHIVE_RATIO=0; UPLOAD_RATIO=0
CREDITS_LENGTH = 8 if ENABLE_CREDITS else 0
CONTENT_LENGTH = TOTAL_VIDEO_LENGTH - CREDITS_LENGTH
NARRATION_LENGTH = CONTENT_LENGTH
NUM_TOTAL_SCENES = max(2, int(CONTENT_LENGTH / SECONDS_PER_SCENE))
WORDS_PER_SCENE = SECONDS_PER_SCENE * 2.5

# Scene counts: 0-ratio = 0 scenes, AI gets remainder
NUM_ARCHIVE_SCENES = int(NUM_TOTAL_SCENES * ARCHIVE_RATIO) if ARCHIVE_RATIO > 0 else 0
NUM_UPLOAD_SCENES = int(NUM_TOTAL_SCENES * UPLOAD_RATIO) if UPLOAD_RATIO > 0 else 0
NUM_AI_SCENES = max(1, NUM_TOTAL_SCENES - NUM_ARCHIVE_SCENES - NUM_UPLOAD_SCENES)
while NUM_AI_SCENES + NUM_ARCHIVE_SCENES + NUM_UPLOAD_SCENES > NUM_TOTAL_SCENES:
    if NUM_ARCHIVE_SCENES > 0: NUM_ARCHIVE_SCENES -= 1
    elif NUM_UPLOAD_SCENES > 0: NUM_UPLOAD_SCENES -= 1
    else: break

NUM_CHAPTERS = max(1, min(3, NUM_TOTAL_SCENES // 3))
SECTIONS_PER_CHAPTER = max(1, NUM_TOTAL_SCENES // NUM_CHAPTERS)

# Interleave scene order (proportional-need algorithm)
scene_types = []
ai_p = 0; ar_p = 0; up_p = 0
for i in range(NUM_TOTAL_SCENES):
    remaining = max(1, NUM_TOTAL_SCENES - i)
    ai_need = (NUM_AI_SCENES - ai_p) / remaining
    ar_need = (NUM_ARCHIVE_SCENES - ar_p) / remaining
    up_need = (NUM_UPLOAD_SCENES - up_p) / remaining
    if ar_need >= ai_need and ar_need >= up_need and ar_p < NUM_ARCHIVE_SCENES:
        scene_types.append('archive'); ar_p += 1
    elif up_need >= ai_need and up_p < NUM_UPLOAD_SCENES:
        scene_types.append('upload'); up_p += 1
    else:
        scene_types.append('ai_generated'); ai_p += 1

if not SESSION_NAME:
    SESSION_NAME = '-'.join(re.sub(r'[^a-zA-Z0-9 ]', '', TOPIC.lower()).split()[:4])
SESSION_DIR = BASE_DIR / SESSION_NAME
for d in ['', 'videos', 'audio', 'archive_clips', 'uploads']:
    (SESSION_DIR / d).mkdir(parents=True, exist_ok=True)

plan = dict(topic=TOPIC, sources=SOURCES, session_name=SESSION_NAME,
    total_video_length=TOTAL_VIDEO_LENGTH, seconds_per_scene=SECONDS_PER_SCENE,
    content_length=CONTENT_LENGTH, credits_length=CREDITS_LENGTH,
    narration_length=NARRATION_LENGTH, ai_ratio=AI_RATIO,
    archive_ratio=ARCHIVE_RATIO, upload_ratio=UPLOAD_RATIO,
    num_total_scenes=NUM_TOTAL_SCENES, num_ai_scenes=NUM_AI_SCENES,
    num_archive_scenes=NUM_ARCHIVE_SCENES, num_upload_scenes=NUM_UPLOAD_SCENES,
    words_per_scene=WORDS_PER_SCENE, scene_types=scene_types,
    num_chapters=NUM_CHAPTERS, sections_per_chapter=SECTIONS_PER_CHAPTER,
    enable_credits=ENABLE_CREDITS, enable_subtitles=ENABLE_SUBTITLES,
    enable_music=ENABLE_MUSIC, enable_effects=ENABLE_EFFECTS)
with open(SESSION_DIR / 'video_plan.json', 'w') as f: json.dump(plan, f, indent=2)

print(f'Session: {SESSION_NAME}')
print(f'Plan: {TOTAL_VIDEO_LENGTH}s = {CONTENT_LENGTH}s content + {CREDITS_LENGTH}s credits')
print(f'   AI: {NUM_AI_SCENES} scenes')
if NUM_ARCHIVE_SCENES > 0: print(f'   Archive: {NUM_ARCHIVE_SCENES} scenes')
if NUM_UPLOAD_SCENES > 0: print(f'   Upload: {NUM_UPLOAD_SCENES} scenes')
print(f'   Narration: {NARRATION_LENGTH:.0f}s ({int(NARRATION_LENGTH*2.5)} words)')
print(f'\nScene order: {scene_types}')
print(f'\n[OK] Plan saved')

---
*ROTBOTS Workshop -- LI-MA 2026*


---

# ==== 02: Script Writer


# [AI] ROTBOTS -- Script Generator
## From Sources to Storyboard

---

1. **Feed the Machine** -- Paste URLs or text, scrape and summarize
2. **The Script Writer** -- LLM generates essay with narration + visual directions
3. **Visual Translation** -- Storyboard with styles and T2V prompts

**You don't need to understand the code.** Just fill in inputs and press > Play.

---

In [ ]:
# ============================================================
# SETUP
# ============================================================
!pip install -q requests beautifulsoup4 pymupdf

import json, re, random, requests, time as _time
from pathlib import Path
from bs4 import BeautifulSoup
from IPython.display import display, Markdown, HTML

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
BASE_DIR.mkdir(parents=True, exist_ok=True)
print('[OK] Setup complete')

In [ ]:
# ============================================================
# API KEY (starts with gsk_)
# Get free: https://console.groq.com/keys
# ============================================================
GROQ_API_KEY = ''  # <-- Paste key here

LLM_MODEL = 'llama-3.3-70b-versatile'
LLM_TEMPERATURE = 0.8
LLM_MAX_TOKENS = 4096
GROQ_API_URL = 'https://api.groq.com/openai/v1/chat/completions'  # NOT here

if not GROQ_API_KEY: print('[!]  Paste your Groq API key above!')
elif not GROQ_API_KEY.startswith('gsk_'): print('[!]  Key should start with gsk_')
else: print(f'[OK] API key configured ({LLM_MODEL})')

In [ ]:
# HELPER FUNCTIONS
def progress_bar(current, total, label='', width=30):
    pct = current / total if total > 0 else 0
    filled = int(width * pct)
    bar = '#' * filled + '-' * (width - filled)
    return f'<div style="font-family:monospace;font-size:14px;padding:2px 0;">{bar} {current}/{total} {label} ({pct:.0%})</div>'

def progress_html(title, current, total, label='', detail='', color='#4ecca3'):
    return (
        f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;">' +
        f'<div style="font-size:16px;font-weight:bold;color:{color};">{title}</div>' +
        progress_bar(current, total, label) +
        (f'<div style="color:#a0a0a0;font-size:12px;margin-top:4px;">{detail}</div>' if detail else '') +
        '</div>'
    )

def query_llm(prompt, system_prompt=None, temperature=None):
    messages = []
    if system_prompt: messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})
    payload = {'model': LLM_MODEL, 'messages': messages, 'temperature': temperature or LLM_TEMPERATURE, 'max_tokens': LLM_MAX_TOKENS}
    response = requests.post(GROQ_API_URL, headers={'Authorization': f'Bearer {GROQ_API_KEY}', 'Content-Type': 'application/json'}, json=payload, timeout=120)
    response.raise_for_status()
    content = response.json()['choices'][0]['message']['content']
    if '<think>' in content and '</think>' in content: content = content.split('</think>')[-1].strip()
    return content

def parse_json_response(response):
    response = response.strip()
    response = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', response)
    if response.startswith('```'):
        lines = response.split('\n')
        response = '\n'.join(lines[1:-1] if lines[-1].strip() == '```' else lines[1:])
    response = response.strip()
    if not response.startswith('[') and not response.startswith('{'):
        for ch in ['{', '[']:
            idx = response.find(ch)
            if idx != -1: response = response[idx:]; break
    for end_char in ['}', ']']:
        last_idx = response.rfind(end_char)
        if last_idx != -1:
            truncated = response[:last_idx + 1]
            for text in [truncated, re.sub(r',\s*([}\]])', r'\1', truncated)]:
                try: return json.loads(text, strict=False)
                except json.JSONDecodeError: pass
    return json.loads(re.sub(r',\s*([}\]])', r'\1', response), strict=False)

def fetch_website_text(url, max_chars=10000):
    response = requests.get(url.strip(), headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    for el in soup(['script','style','nav','header','footer','aside','form']): el.decompose()
    main = None
    for sel in ['article','main','[role="main"]','.content','#content']:
        main = soup.select_one(sel)
        if main: break
    text = main.get_text(separator=' ',strip=True) if main else (soup.find('body') or soup).get_text(separator=' ',strip=True)
    return re.sub(r'\s+', ' ', text).strip()[:max_chars]

def fetch_pdf_text(source, max_chars=10000):
    import tempfile
    try: import pymupdf as fitz
    except ImportError: import fitz
    temp_file = None
    try:
        if source.startswith('http'):
            resp = requests.get(source, headers={'User-Agent':'Mozilla/5.0'}, timeout=60); resp.raise_for_status()
            temp_file = tempfile.NamedTemporaryFile(suffix='.pdf', delete=False); temp_file.write(resp.content); temp_file.close()
            pdf_path = temp_file.name
        else: pdf_path = source
        doc = fitz.open(pdf_path); parts = []; total = 0
        for page in doc: t = page.get_text(); parts.append(t); total += len(t);
        doc.close()
        return re.sub(r'\n{3,}','\n\n','\n'.join(parts))[:max_chars]
    finally:
        if temp_file: import os; os.unlink(temp_file.name)

print('[OK] Helpers loaded')

---
## [DL] Station 1: Feed the Machine

In [ ]:
# SCRAPE + SUMMARIZE with live progress
prog = display(HTML(''), display_id=True)
source_texts = []
total_steps = len(SOURCES) * 2  # scrape + summarize each
step = 0
t0 = _time.time()

for i, source in enumerate(SOURCES):
    prog.update(HTML(progress_html(f'[DL] Scraping source {i+1}/{len(SOURCES)}', step, total_steps, 'steps', source[:60])))
    try:
        if source.startswith('http') and source.lower().endswith('.pdf'): text = fetch_pdf_text(source)
        elif source.startswith('http'): text = fetch_website_text(source)
        else: text = source
        source_texts.append({'source': source[:100], 'text': text})
    except Exception as e: pass
    step += 1

summaries = []
for i, src in enumerate(source_texts):
    prog.update(HTML(progress_html(f'[BRAIN] Summarizing source {i+1}/{len(source_texts)}', step, total_steps, 'steps', f'LLM call... ({src["source"][:40]})')))
    summary = query_llm(f'Summarize in 2-3 short paragraphs for a video essay about "{TOPIC}":\n\n{src["text"][:6000]}', system_prompt='Be concise.')
    summaries.append({'source': src['source'], 'summary': summary})
    step += 1

with open(SESSION_DIR / 'summaries.json', 'w') as f: json.dump({'topic': TOPIC, 'sources': summaries}, f, indent=2)
prog.update(HTML(progress_html(f'[OK] {len(summaries)} sources scraped & summarized in {_time.time()-t0:.1f}s', total_steps, total_steps, 'steps')))

In [ ]:
# VIEW
for i, s in enumerate(summaries):
    display(Markdown(f'### Source {i+1}: {s["source"]}'))
    display(Markdown(s['summary'] + '\n\n---'))

---
## >> Station 2: The Script Writer

In [ ]:
# OUTLINE + ESSAY NARRATION
total_llm = 3 + NUM_CHAPTERS
prog = display(HTML(''), display_id=True)
t0 = _time.time()

# Step 1: Outline
prog.update(HTML(progress_html('Generating outline...', 0, total_llm, 'LLM calls')))
summaries_text = '\n\n'.join([f'SOURCE: {s["source"]}\n{s["summary"]}' for s in summaries])
outline_prompt = f'Create an essay outline for a {TARGET_VIDEO_LENGTH}s video about: "{TOPIC}"\n\nRESEARCH:\n{summaries_text}\n\nExactly {NUM_CHAPTERS} chapters. JSON only:\n{{"title": "...", "thesis": "...", "chapters": [{{"chapter": 1, "title": "...", "summary": "...", "key_points": ["..."]}}]}}'
for attempt in range(3):
    try: outline = parse_json_response(query_llm(outline_prompt, 'Expert essay writer. Concise.')); break
    except Exception as e: prog.update(HTML(progress_html(f'Retry {attempt+1}/3...', 0, total_llm, 'LLM calls', str(e)[:60], '#f39c12')))
if len(outline.get('chapters',[])) > NUM_CHAPTERS: outline['chapters'] = outline['chapters'][:NUM_CHAPTERS]

# Step 2: Full essay narration
prog.update(HTML(progress_html('Writing full essay narration...', 1, total_llm, 'LLM calls')))
chapter_summaries = '\n'.join([f'Ch {c["chapter"]}: {c["title"]} - {c.get("summary","")}' for c in outline['chapters']])
essay_prompt = f'''Write a single continuous narration for a {TARGET_VIDEO_LENGTH}-second video about: "{TOPIC}"

STRUCTURE:
{chapter_summaries}

RULES:
- Write EXACTLY {TOTAL_NARRATION_WORDS} words total
- One flowing essay, NO section breaks or headers
- Engaging, documentary-style prose
- Write complete sentences, not fragments
- Output ONLY the essay text, nothing else'''
essay_narration = query_llm(essay_prompt, f'Expert video essay scriptwriter. Write exactly {TOTAL_NARRATION_WORDS} words of flowing narration.').strip()
wc = len(essay_narration.split())
while wc < int(TOTAL_NARRATION_WORDS * 0.9):
    needed = TOTAL_NARRATION_WORDS - wc
    print(f'  {wc} words -- expanding by {needed}...')
    more = query_llm(f'Continue this essay with EXACTLY {needed} more words. Same style, same topic:\n\n{essay_narration[-500:]}', f'Continue seamlessly. Write exactly {needed} words.').strip()
    essay_narration = essay_narration + ' ' + more
    wc = len(essay_narration.split())

# Step 3: Visual directions per chapter
for i, ch in enumerate(outline['chapters']):
    prog.update(HTML(progress_html(f'Visual directions: Ch {ch.get("chapter",i+1)}', 2+i, total_llm, 'LLM calls')))
    vis_prompt = f'Write {SECTIONS_PER_CHAPTER} visual scene descriptions for chapter "{ch["title"]}" of a video about "{TOPIC}". JSON: [{{"section": 1, "visual_direction": "describe the visual", "mood": "emotional tone"}}]'
    try:
        sections = parse_json_response(query_llm(vis_prompt, 'Visual director. Concise scene descriptions.'))
        if isinstance(sections, dict):
            for v in sections.values():
                if isinstance(v, list): sections = v; break
            else: sections = [sections]
    except: sections = [{'section': 1, 'visual_direction': ch.get('summary',''), 'mood': 'neutral'}]
    outline['chapters'][i]['sections'] = sections[:SECTIONS_PER_CHAPTER]

# Save
outline['narration'] = essay_narration
outline['credits'] = {'sources': [s['source'] for s in summaries]}
with open(SESSION_DIR / 'essay_script.json', 'w') as f: json.dump(outline, f, indent=2)
wc = len(essay_narration.split())
prog.update(HTML(progress_html(f'"{outline.get("title","")}" -- {wc} words ~ {wc/2.5:.0f}s ({_time.time()-t0:.1f}s)', total_llm, total_llm, 'LLM calls')))


In [ ]:
# VIEW
display(Markdown(f'# {outline.get("title","")}\n*{outline.get("thesis","")}*\n\n---'))
display(Markdown(f'## Full Narration ({len(essay_narration.split())} words)\n\n> {essay_narration}\n\n---'))
for ch in outline['chapters']:
    display(Markdown(f'### Ch {ch["chapter"]}: {ch["title"]}'))
    for s in ch.get('sections',[]):
        display(Markdown(f'**Sec {s.get("section","?")}** *{s.get("mood","?")}* -- {s.get("visual_direction","")}'))


---
## [SCENE] Station 3: Visual Translation

In [ ]:
# STYLE ARC
STYLE_ARCS = {
    'calm_to_dramatic': {'description': 'Calm -> intense', 'early': ['documentary','nature'], 'middle': ['news_report','sports_commentary'], 'late': ['action_movie','horror']},
    'documentary_journey': {'description': 'Documentary, increasing energy', 'early': ['documentary'], 'middle': ['nature','news_report'], 'late': ['action_movie','sports_commentary']},
    'chaos_arc': {'description': 'Brainrot chaos', 'early': ['classic_brainrot','zoomer_brainrot'], 'middle': ['surreal_brainrot','hyperpop_brainrot'], 'late': ['fever_dream_brainrot','cursed_brainrot']},
}
STYLES = {
    'documentary':{'vk':'cinematic, professional lighting','ak':'ambient sounds'},
    'nature':{'vk':'wide nature shots, golden hour','ak':'nature sounds, wind'},
    'news_report':{'vk':'news studio, professional','ak':'news audio, serious'},
    'sports_commentary':{'vk':'dynamic angles, slow motion','ak':'crowd, energetic'},
    'action_movie':{'vk':'dynamic movement, dramatic lighting','ak':'orchestral hits'},
    'horror':{'vk':'dark lighting, shadows, fog','ak':'creepy ambience'},
    'classic_brainrot':{'vk':'chaotic, deep fried','ak':'bass boosted'},
    'surreal_brainrot':{'vk':'dreamlike, impossible geometry','ak':'slowed reverb'},
    'zoomer_brainrot':{'vk':'meme aesthetic, ironic','ak':'TikTok sounds'},
    'hyperpop_brainrot':{'vk':'maximalist, Y2K','ak':'hyperpop beats'},
    'fever_dream_brainrot':{'vk':'hallucinatory, warped','ak':'echoing'},
    'cursed_brainrot':{'vk':'unsettling, jpeg artifacts','ak':'distorted'},
}

CHOSEN_ARC = 'calm_to_dramatic'
arc = STYLE_ARCS[CHOSEN_ARC]
print(f'[ART] {CHOSEN_ARC}: {arc["description"]}')

In [ ]:
# STORYBOARD + STYLES
all_sec = []
for ch in outline.get('chapters', []):
    for sec in ch.get('sections', []):
        d = dict(**sec)
        d['chapter_title'] = ch['title']
        d['chapter'] = ch.get('chapter', 0)
        all_sec.append(d)

scenes = []
sn = 1; si = 0
for stype in scene_types:
    if si < len(all_sec):
        sec = all_sec[si]
    else:
        sec = dict(narration='', visual_direction='', mood='neutral', chapter_title=TOPIC, section=si+1)
    scenes.append(dict(
        scene=sn, scene_type=stype,
        title=str(sec.get('chapter_title','')) + ' - Part ' + str(sec.get('section', si+1)),
        visual_direction=sec.get('visual_direction', ''),
        mood=sec.get('mood', ''),
        duration=SECONDS_PER_SCENE))
    sn += 1; si += 1

ai_sc = [s for s in scenes if s['scene_type'] == 'ai_generated']
total_ai = len(ai_sc)
ee = max(1, int(total_ai * 0.25))
ls = max(ee + 1, int(total_ai * 0.75))
last = None
for i, sc in enumerate(ai_sc):
    phase = 'early' if i < ee else ('late' if i >= ls else 'middle')
    pool = arc.get(phase, ['documentary'])
    avail = [s for s in pool if s != last] or pool
    st = random.choice(avail)
    sc['assigned_style'] = st
    sc['visual_keywords'] = STYLES.get(st, {}).get('vk', '') if isinstance(STYLES.get(st), dict) else STYLES.get(st, '')
    last = st

with open(SESSION_DIR / 'storyboard.json', 'w') as f: json.dump(scenes, f, indent=2)
for s in scenes:
    tag = f' [{s.get("assigned_style","")}]' if s.get('assigned_style') else ''
    icon = {'ai_generated':'[AI]','archive':'[ARCH]','upload':'[UP]'}.get(s['scene_type'],'*')
    print(f'   {icon} Scene {s["scene"]}: [{s["scene_type"]}] {s["title"]}{tag}')

In [ ]:
# T2V PROMPTS (AI scenes only)
ai_scenes_list = [sc for sc in scenes if sc['scene_type'] == 'ai_generated']
if not ai_scenes_list:
    print(' No AI scenes in plan! Check ARCHIVE_RATIO / UPLOAD_RATIO.')
OPENINGS = ['Start with SHOT TYPE','Start with ACTION','Start with ENVIRONMENT','Start with LIGHTING','Start with CAMERA']
prompts = []
prog = display(HTML(''), display_id=True)
t0 = _time.time()

for idx, sc in enumerate(ai_scenes_list):
    st = sc.get('assigned_style','documentary')
    vk = sc.get('visual_keywords','')
    prog.update(HTML(progress_html(f'Scene {sc["scene"]}: {sc["title"]}', idx, len(ai_scenes_list), 'prompts', f'Style: {st}')))
    sys_p = f'T2V prompt expert. ONE paragraph, 3-5 sentences for {SECONDS_PER_SCENE}s clip. Style: {st}. Visual: {vk}. No text overlays.'
    user_p = f'T2V prompt for {SECONDS_PER_SCENE}s:\nVisual: {sc.get("visual_direction","")}\nMood: {sc.get("mood","")}\n{random.choice(OPENINGS)}\nOutput ONLY the prompt text.'
    t2v = query_llm(user_p, sys_p).strip().strip('"')
    prompts.append(dict(scene=sc['scene'], title=sc['title'], t2v_prompt=t2v,

with open(SESSION_DIR / 'prompts.json', 'w') as f: json.dump(prompts, f, indent=2)
prog.update(HTML(progress_html(f'[OK] {len(prompts)} T2V prompts in {_time.time()-t0:.1f}s', len(ai_scenes_list), len(ai_scenes_list), 'prompts')))

In [ ]:
# VIEW
display(Markdown(f'# [SCENE] {len(prompts)} scenes × ~{SECONDS_PER_SCENE}s = ~{len(prompts)*SECONDS_PER_SCENE}s\n\n---'))
for p in prompts:
    wc=len(p['narration'].split())
    display(Markdown(f'### Scene {p["scene"]}: {p["title"]}\n`{p["style"]}` | {p["duration"]}s | {wc}w\n\n> [MIC] {p["narration"]}\n\n> [?] {p["t2v_prompt"]}\n\n---'))
print(f'[DIR] Session "{SESSION_NAME}" saved to Google Drive')

---
*ROTBOTS Workshop -- LI-MA 2026*


---

# ==== 03: Archive Scraper


# [CITY] ROTBOTS -- Archive Scraper

---

Downloads and segments footage from archive.org. Skipped if ARCHIVE_RATIO = 0.

> **[?]** Found footage: who chose it? You or the machine?

---

In [ ]:
# SETUP
!pip install -q yt-dlp
import json, re, subprocess
from pathlib import Path
from IPython.display import display, Markdown
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
print('[OK] Setup')

---
## [CITY] Archive Sources

Add archive.org URLs or item IDs below.

In [ ]:
# ARCHIVE CONFIG
ARCHIVE_URLS = [
    # 'https://archive.org/details/example_item',
]

PREFER_HEIGHT = 480
MAX_CLIP_DURATION = 5  # match SECONDS_PER_SCENE
MIN_CLIP_DURATION = 3
MAX_EXTRACT_DURATION = 180

In [ ]:
# SCRAPE
def parse_archive_id(url):
    url = url.strip().rstrip('/')
    m = re.search(r'archive\.org/(?:details|download)/([^/?#]+)', url)
    if m: return m.group(1)
    if '/' not in url and '.' not in url: return url
    raise ValueError('Cannot parse: ' + url)

def get_dur(path):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)], capture_output=True, text=True, timeout=10).stdout.strip())
    except: return 0

archive_clips = []
ARCHIVE_DIR = SESSION_DIR / 'archive_clips'; ARCHIVE_DIR.mkdir(exist_ok=True)
if ARCHIVE_URLS and NUM_ARCHIVE_SCENES > 0:
    ATEMP = Path('/content/temp_archive'); ATEMP.mkdir(exist_ok=True)
    for i, url in enumerate(ARCHIVE_URLS):
        try: aid = parse_archive_id(url)
        except ValueError as e: print(f'Error: {e}'); continue
        print(f'Downloading [{i+1}/{len(ARCHIVE_URLS)}] {aid}')
        out_tpl = str(ATEMP / (aid + '.%(ext)s'))
        try: subprocess.run(['yt-dlp', 'https://archive.org/details/' + aid, '-f', 'bestvideo[height<=' + str(PREFER_HEIGHT) + ']+bestaudio/best[height<=' + str(PREFER_HEIGHT) + ']/best', '--merge-output-format', 'mp4', '--no-playlist', '--no-warnings', '-o', out_tpl, '--quiet'], check=True, timeout=600)
        except: print('  Download failed'); continue
        video = None
        for fp in ATEMP.iterdir():
            if fp.stem == aid and fp.suffix in ('.mp4','.mkv','.webm'): video = fp; break
        if not video: continue
        total = get_dur(video); clip_dir = ARCHIVE_DIR / aid; clip_dir.mkdir(exist_ok=True)
        extract_end = min(total, MAX_EXTRACT_DURATION) if MAX_EXTRACT_DURATION > 0 else total
        t = 0; idx2 = 0
        while t < extract_end:
            clip_dur = min(MAX_CLIP_DURATION, extract_end - t)
            if clip_dur < MIN_CLIP_DURATION: break
            clip_out = clip_dir / ('archive_' + f'{idx2:03d}' + '.mp4')
            r = subprocess.run(['ffmpeg','-y','-ss',str(t),'-i',str(video),'-t',str(clip_dur),'-c:v','libx264','-preset','fast','-crf','23','-an',str(clip_out)], capture_output=True, text=True, timeout=120)
            if r.returncode == 0 and clip_out.exists():
                archive_clips.append(dict(path=str(clip_out), duration=round(clip_dur,1), archive_id=aid)); idx2 += 1
            t += clip_dur
        video.unlink(missing_ok=True); print(f'  {idx2} clips')
    with open(SESSION_DIR / 'archive_clips.json', 'w') as f: json.dump(dict(clips=archive_clips, total=len(archive_clips)), f, indent=2)
    print(f'[OK] {len(archive_clips)} archive clips')
elif NUM_ARCHIVE_SCENES > 0: print(f' Need {NUM_ARCHIVE_SCENES} archive clips but no ARCHIVE_URLS!')
else: print('No archive footage needed')

---
*ROTBOTS Workshop -- LI-MA 2026*


---

# ==== 04: Upload Footage


# [DIR] ROTBOTS -- Upload Footage

---

Upload your own video clips to mix with AI-generated footage.

> **[?]** What's the difference between footage you filmed and footage a machine generated?

---

---
## [?] Upload Your Clips

In [ ]:
# UPLOAD
print('[?] Upload MP4 files:')
uploaded = files.upload()
upload_clips = []
for fn, content in uploaded.items():
    dest = UPLOADS_DIR / fn
    with open(dest,'wb') as f: f.write(content)
    upload_clips.append({'path':str(dest),'filename':fn,'size':len(content)})
    print(f'   [OK] {fn} ({len(content)//1024}KB)')
with open(SESSION_DIR/'upload_manifest.json','w') as f: json.dump({'clips':upload_clips,'total':len(upload_clips)},f,indent=2)
print(f'\n[OK] {len(upload_clips)} clips uploaded')
if needed>0 and len(upload_clips)<needed: print(f'   [!] Plan needs {needed} but only {len(upload_clips)} uploaded')

In [ ]:
# PREVIEW
existing = list(UPLOADS_DIR.glob('*.mp4'))
if existing:
    for f in existing[:6]:
        display(Markdown(f'### {f.name}'))
        try: display(Video(str(f),embed=True,width=480))
        except: print(f'   {f}')
else: print('No uploads yet.')

---
*ROTBOTS Workshop -- LI-MA 2026*


---

# ==== 05: Effects and Log


# [ART] ROTBOTS -- Effects & Chain Log

---

Assign FFmpeg effects per AI scene. View the chain of AI decisions.

Skip if `ENABLE_EFFECTS = False` in your plan.

---

In [ ]:
# EFFECTS
EFFECTS = {'none':'No effect','film_grain':'Subtle grain','vhs_artifacts':'VHS tracking','celluloid_scratches':'Film scratches','sepia_tone':'Warm sepia','bw_transition':'Black & white','color_grade_warm':'Warm orange','color_grade_cool':'Cool blue','vignette':'Dark edges','flicker':'Projector flicker','desaturate':'Muted colors'}
for k,v in EFFECTS.items(): print(f'   {k:25s} {v}')

In [ ]:
# ASSIGN EFFECTS
EFFECT_MODE = 'random'
PER_SCENE_EFFECTS = {}
EFFECT_INTENSITY = 0.7
pf = SESSION_DIR / 'prompts.json'
if not pf.exists(): print('[!!] No prompts.json!')
elif not plan.get('enable_effects',True): print('[ART] Effects disabled.')
else:
    with open(pf) as f: prompts = json.load(f)
    enames = [k for k in EFFECTS if k!='none']
    for p in prompts:
        sn=p['scene']
        if sn in PER_SCENE_EFFECTS: p['ffmpeg_effects']=[PER_SCENE_EFFECTS[sn]] if PER_SCENE_EFFECTS[sn]!='none' else []
        elif EFFECT_MODE=='random': p['ffmpeg_effects']=[random.choice(enames)]
        elif EFFECT_MODE!='none': p['ffmpeg_effects']=[EFFECT_MODE]
        else: p['ffmpeg_effects']=[]
        p['effect_intensity']=EFFECT_INTENSITY
    with open(pf,'w') as f: json.dump(prompts,f,indent=2)
    print('[OK] Effects:')
    for p in prompts: print(f'   Scene {p["scene"]}: {p.get("ffmpeg_effects",[])} ({EFFECT_INTENSITY:.0%})')

---
## [?] Chain Log

In [ ]:
# CHAIN LOG
chain={'session':SESSION_NAME,'config':plan,'interactions':[]}
if (SESSION_DIR/'summaries.json').exists():
    with open(SESSION_DIR/'summaries.json') as f: d=json.load(f)
    for s in d.get('sources',[]): chain['interactions'].append({'stage':'summarize','purpose':s['source'][:40],'length':len(s['summary'])})
if (SESSION_DIR/'essay_script.json').exists():
    with open(SESSION_DIR/'essay_script.json') as f: e=json.load(f)
    chain['interactions'].append({'stage':'outline','purpose':e.get('title','')})
    for ch in e.get('chapters',[]):
        for sec in ch.get('sections',[]): chain['interactions'].append({'stage':'chapter','purpose':f'Ch{ch["chapter"]}.{sec.get("section","?")}'})
if (SESSION_DIR/'prompts.json').exists():
    with open(SESSION_DIR/'prompts.json') as f: pd=json.load(f)
    for p in pd: chain['interactions'].append({'stage':'t2v','purpose':f'Scene {p["scene"]}','effect':p.get('ffmpeg_effects',[])})
with open(SESSION_DIR/'chain_log.json','w') as f: json.dump(chain,f,indent=2)
print(f'[?] {len(chain["interactions"])} decisions')
for i in chain['interactions']:
    eff=f' [ART]{i["effect"][0]}' if i.get('effect') else ''
    print(f'   [{i["stage"]}] {i["purpose"]}{eff}')

---
*ROTBOTS Workshop -- LI-MA 2026*


---

# ==== 06: The Voice


# [MIC] ROTBOTS -- The Voice

---

Narration for **all** scenes (AI + found footage). The voice runs over everything.

> **[?]** What does a synthetic voice do to trust?

---

In [ ]:
# SETUP
!pip install -q edge-tts
import json, subprocess, time as _time, edge_tts
from pathlib import Path
from IPython.display import display, Markdown, Audio, HTML
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
def progress_bar(c,t,l='',w=30):
    p=c/t if t>0 else 0;f=int(w*p);return f'<div style="font-family:monospace;font-size:14px;padding:2px 0;">{"#"*f}{"-"*(w-f)} {c}/{t} {l} ({p:.0%})</div>'
def progress_html(title,c,t,l='',d='',color='#4ecca3'):
    return f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;"><div style="font-size:16px;font-weight:bold;color:{color};">{title}</div>{progress_bar(c,t,l)}'+(f'<div style="color:#a0a0a0;font-size:12px;margin-top:4px;">{d}</div>' if d else '')+'</div>'
print('[OK] Setup')

In [ ]:
# LOAD ESSAY NARRATION
AUDIO_DIR = SESSION_DIR / 'audio'; AUDIO_DIR.mkdir(exist_ok=True)
essay_file = SESSION_DIR / 'essay_script.json'
if essay_file.exists():
    with open(essay_file) as f: essay = json.load(f)
    narration_text = essay.get('narration', '')
    if narration_text:
        wc = len(narration_text.split())
        print(f'Full essay narration: {wc} words ~ {wc/2.5:.0f}s')
    else:
        print('[!!] No narration field in essay_script.json!')
else:
    print('[!!] No essay_script.json!')
    narration_text = ''


In [ ]:
# VOICE
VOICES={'female_us':'en-US-AriaNeural','male_us':'en-US-GuyNeural','female_uk':'en-GB-SoniaNeural','male_uk':'en-GB-RyanNeural','documentary':'en-US-GuyNeural','dramatic':'en-GB-RyanNeural','energetic':'en-US-JennyNeural'}
CHOSEN_VOICE='documentary'
voice_name=VOICES[CHOSEN_VOICE]
print(f'[MIC] {CHOSEN_VOICE} ({voice_name})')

In [ ]:
# GENERATE SINGLE NARRATION
async def gen_tts(text, path, voice, rate='+0%'):
    await edge_tts.Communicate(text, voice, rate=rate).save(str(path))
def get_dur(path):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)],capture_output=True,text=True,timeout=10).stdout.strip())
    except: return 5.0

prog = display(HTML(''), display_id=True)
t0 = _time.time()
out = AUDIO_DIR / 'narration_full.mp3'
audio_file = None

if narration_text:
    prog.update(HTML(progress_html('Generating full narration...', 0, 1, 'TTS')))
    try:
        await gen_tts(narration_text, out, voice_name)
        audio_duration = get_dur(out)
        audio_file = {'path': str(out), 'duration': audio_duration}
        print(f'   narration_full.mp3: {audio_duration:.1f}s')
    except Exception as e:
        print(f'   [!!] {e}')
else:
    print('No narration text to generate!')

manifest = {'voice': voice_name, 'file': audio_file}
with open(SESSION_DIR / 'audio_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

if audio_file:
    prog.update(HTML(progress_html(f'[OK] Narration: {audio_file["duration"]:.1f}s ({_time.time()-t0:.1f}s)', 1, 1, 'TTS')))


In [ ]:
# PREVIEW
if audio_file and Path(audio_file['path']).exists():
    display(Markdown(f'### Full Narration ({audio_file["duration"]:.1f}s)'))
    display(Audio(audio_file['path'], autoplay=False))
else:
    print('No audio to preview')


---
*ROTBOTS Workshop -- LI-MA 2026*


---

# ==== 07: Generate


# [AI] ROTBOTS -- AI Video Generator

---

Generates clips **only for AI scenes** (not archive/upload).

> **[?]** What does it cost to generate these clips?

---

In [ ]:
# SETUP
!pip install -q fal-client requests
import fal_client,json,os,time as _time,requests,threading
from pathlib import Path
from IPython.display import display,Markdown,Video,HTML
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR=Path('/content/drive/MyDrive/rotbots_workshop')
print('[OK] Setup')

In [ ]:
# API KEY
FAL_API_KEY=''
if not FAL_API_KEY: print('[!] Paste fal.ai key!')
else: os.environ['FAL_KEY']=FAL_API_KEY; print('[OK] fal.ai')

In [ ]:
# PROMPTS
pf=SESSION_DIR/'prompts.json'
if pf.exists():
    with open(pf) as f: prompts=json.load(f)
    print(f'[OK] {len(prompts)} AI prompts')
    for p in prompts: print(f'   Scene {p["scene"]}: {p["title"]} ({p["duration"]}s)')
else: print('[!!] No prompts.json!')

In [ ]:
# MODEL
MODELS={'wan-t2v':{'endpoint':'fal-ai/wan-t2v','desc':'Wan 2.1','ds':5},'minimax':{'endpoint':'fal-ai/minimax/video-01','desc':'MiniMax','ds':6},'ltx-video':{'endpoint':'fal-ai/ltx-video/v2.1','desc':'LTX 2.1','ds':5}}
CHOSEN_MODEL='wan-t2v'
model=MODELS[CHOSEN_MODEL]
print(f'[AI] {CHOSEN_MODEL}: {model["desc"]}')

In [ ]:
# GENERATE
prog=display(HTML(''),display_id=True);generated_videos=[];t0=_time.time();_stop=False
def _timer(ph,si,done,total,ts):
    sp=['|','/','-','\','|','/','-','\','|','/'];tick=0
    while not _stop:
        el=_time.time()-ts;ce=_time.time()-si.get('cs',ts);avg=el/done[0] if done[0]>0 else 0;eta=(total-done[0]-1)*avg+max(0,avg-ce) if avg>0 else 0
        s=sp[tick%len(sp)];p=done[0]/total if total>0 else 0;fl=int(30*p)
        try:ph.update(HTML(f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;"><div style="font-size:16px;font-weight:bold;color:#e94560;">{s} Scene {si.get("sn","?")}: {si.get("title","")}</div><div style="font-size:14px;padding:2px 0;">{"#"*fl}{"-"*(30-fl)} {done[0]}/{total} ({p:.0%})</div><div style="color:#a0a0a0;font-size:12px;">T: {el:.0f}s | clip: {ce:.0f}s | ~{eta:.0f}s left</div></div>'))
        except:pass
        tick+=1;_time.sleep(1)
for i,pd in enumerate(prompts):
    sn=pd['scene'];dur=pd.get('duration',model['ds'])
    si={'sn':sn,'title':pd['title'],'cs':_time.time()};done=[len(generated_videos)]
    _stop=False;t=threading.Thread(target=_timer,args=(prog,si,done,len(prompts),t0),daemon=True);t.start()
    try:
        inp={'prompt':pd['t2v_prompt']}
        if 'wan' in CHOSEN_MODEL:inp['aspect_ratio']='16:9';inp['enable_prompt_expansion']=False
        result=fal_client.subscribe(model['endpoint'],arguments=inp);url=None
        if isinstance(result,dict):
            if 'video' in result and isinstance(result['video'],dict):url=result['video'].get('url')
            elif 'video' in result:url=result['video']
            elif 'url' in result:url=result['url']
        if url:
            vp=VIDEOS_DIR/f'scene_{sn:03d}.mp4'
            with open(vp,'wb') as f:f.write(requests.get(url,timeout=120).content)
            generated_videos.append({'scene':sn,'path':str(vp),'duration':dur,'source':'generated'})
    except:pass
    finally:_stop=True;t.join(timeout=2)
el=_time.time()-t0
prog.update(HTML(f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;"><div style="font-size:16px;font-weight:bold;color:#4ecca3;">[OK] {len(generated_videos)}/{len(prompts)} videos in {el:.0f}s</div></div>'))
with open(SESSION_DIR/'video_manifest.json','w') as f:json.dump({'model':CHOSEN_MODEL,'videos':generated_videos},f,indent=2)

In [ ]:
# PREVIEW
for v in sorted(generated_videos,key=lambda x:x['scene']):
    display(Markdown(f'### [AI] Scene {v["scene"]}'))
    try:display(Video(v['path'],embed=True,width=640))
    except:print(v['path'])
    display(Markdown('---'))

---
*ROTBOTS Workshop -- LI-MA 2026*


---

# ==== 08: Subtitles


# [NOTE] ROTBOTS -- Subtitles

---

TikTok-style word-by-word subtitles. Skip if `ENABLE_SUBTITLES = False`.

---

In [ ]:
# SETUP
!pip install -q faster-whisper
import json, random, subprocess, time as _time
from pathlib import Path
from IPython.display import display, Markdown, HTML
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
def progress_bar(c,t,l='',w=30):
    p=c/t if t>0 else 0;f=int(w*p);return f'<div style="font-family:monospace;font-size:14px;padding:2px 0;">{"#"*f}{"-"*(w-f)} {c}/{t} {l} ({p:.0%})</div>'
def progress_html(title,c,t,l='',d='',color='#4ecca3'):
    return f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;"><div style="font-size:16px;font-weight:bold;color:{color};">{title}</div>{progress_bar(c,t,l)}'+(f'<div style="color:#a0a0a0;font-size:12px;margin-top:4px;">{d}</div>' if d else '')+'</div>'
print('[OK] Setup')

In [ ]:
# SETTINGS
SUBTITLE_MODE='random'
MIXUP_MIN=10;MIXUP_MAX=25
STYLES={
    '01':{'font':'Impact','size':80,'bold':-1,'primary':'&H00FFFFFF','outline_c':'&H00000000','back':'&H00000000','border_style':1,'outline':7,'shadow':0,'alignment':5,'margin_v':20,'upper':True,'anim':r'{\fscx0\fscy0\t(0,40,\fscx130\fscy130)\t(40,80,\fscx100\fscy100)}','desc':'Classic Impact'},
    '04':{'font':'Arial Black','size':76,'bold':-1,'primary':'&H00FFFFFF','outline_c':'&H008030FF','back':'&H00000000','border_style':1,'outline':8,'shadow':3,'alignment':5,'margin_v':30,'upper':True,'anim':r'{\fscx0\fscy0\t(0,30,\fscx140\fscy140)\t(30,70,\fscx100\fscy100)}','desc':'Purple bounce'},
    '05':{'font':'Arial Black','size':70,'bold':0,'primary':'&H00FFFFFF','outline_c':'&H00000000','back':'&HCC1A1A1A','border_style':3,'outline':16,'shadow':0,'alignment':2,'margin_v':80,'upper':True,'anim':r'{\fad(80,0)\fscx95\fscy95\t(0,60,\fscx100\fscy100)}','desc':'Dark box'},
    '06':{'font':'Impact','size':85,'bold':0,'primary':'&H0000DDFF','outline_c':'&H000000CC','back':'&H00000000','border_style':1,'outline':7,'shadow':4,'alignment':5,'margin_v':20,'upper':True,'anim':r'{\fscx0\fscy0\t(0,25,\fscx150\fscy150)\t(25,55,\fscx90\fscy90)\t(55,80,\fscx105\fscy105)\t(80,100,\fscx100\fscy100)}','desc':'Orange spring'},
    '08':{'font':'Arial Black','size':72,'bold':0,'primary':'&H00FFFF00','outline_c':'&H00FF0066','back':'&H00000000','border_style':1,'outline':5,'shadow':5,'alignment':5,'margin_v':20,'upper':True,'anim':r'{\fscx0\fscy0\t(0,40,\fscx115\fscy115)\t(40,80,\fscx100\fscy100)}','desc':'Neon pop-in'},
}
for k,v in STYLES.items(): print(f'   {k}: {v["desc"]}')

In [ ]:
# WHISPER
from faster_whisper import WhisperModel
prog = display(HTML(progress_html('Loading Whisper...', 0, 1)), display_id=True)
wm = WhisperModel('base', device='cpu', compute_type='int8')

narration_file = AUDIO_DIR / 'narration_full.mp3'
all_words = []
t0 = _time.time()

if narration_file.exists():
    prog.update(HTML(progress_html('Transcribing narration...', 0, 1, 'files')))
    segs, info = wm.transcribe(str(narration_file), word_timestamps=True, language='en')
    for seg in segs:
        if seg.words:
            for w in seg.words:
                all_words.append({'word': w.word.strip(), 'start': w.start, 'end': w.end})
    prog.update(HTML(progress_html(f'{len(all_words)} words ({_time.time()-t0:.1f}s)', 1, 1, 'files')))
else:
    print('No narration_full.mp3 found!')


In [ ]:
# GENERATE ASS
W=1280;H=720;sc=H/512
def sl(n,s):
    sz=int(s['size']*sc);ol=int(s['outline']*sc);mv=int(s['margin_v']*sc)
    return f'Style: {n},{s["font"]},{sz},{s["primary"]},&H0000FFFF,{s["outline_c"]},{s["back"]},{s["bold"]},0,0,0,100,100,2,0,{s["border_style"]},{ol},{s["shadow"]},{s["alignment"]},10,10,{mv},1'
def fmt(sec):
    h=int(sec//3600);m=int((sec%3600)//60);s=int(sec%60);cs=int((sec%1)*100)
    return f'{h}:{m:02d}:{s:02d}.{cs:02d}'
mx=SUBTITLE_MODE=='random'
sls=[sl(f'TT{k}',v) for k,v in STYLES.items()] if mx else [sl('TT',STYLES.get(SUBTITLE_MODE,STYLES['01']))]
hdr=f'[Script Info]\nTitle: ROTBOTS\nScriptType: v4.00+\nWrapStyle: 0\nScaledBorderAndShadow: yes\nPlayResX: {W}\nPlayResY: {H}\n\n[V4+ Styles]\nFormat: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\n'+'\n'.join(sls)+'\n\n[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n'
keys=list(STYLES.keys());ck=random.choice(keys) if mx else SUBTITLE_MODE if SUBTITLE_MODE in STYLES else '01'
ns=random.uniform(MIXUP_MIN,MIXUP_MAX) if mx else 99999
dlg=[]
for w in all_words:
    if mx and w['start']>=ns: ck=random.choice(keys);ns=w['start']+random.uniform(MIXUP_MIN,MIXUP_MAX)
    sty=STYLES[ck];sn=f'TT{ck}' if mx else 'TT'
    wd=w['word'].replace('\\','\\\\').replace('{','\\{').replace('}','\\}')
    dlg.append(f'Dialogue: 0,{fmt(w["start"])},{fmt(w["end"])},{sn},,0,0,0,,{sty["anim"]}{wd.upper() if sty["upper"] else wd}')
with open(SESSION_DIR/'subtitles.ass','w',encoding='utf-8') as f: f.write(hdr+'\n'.join(dlg))
print(f'[OK] subtitles.ass: {len(dlg)} words')

---
*ROTBOTS Workshop -- LI-MA 2026*


---

# ==== 09: Assemble


# [FILM] ROTBOTS -- Assemble
## Interleaved: AI + Archive + Upload -> Final Video

---

Builds the final video following the storyboard scene order.

---

In [ ]:
# SETUP
import json, subprocess, shutil, os, re
from pathlib import Path
from IPython.display import display, Markdown, Video
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
TEMP = Path('/content/temp_assembly'); TEMP.mkdir(exist_ok=True)
print('[OK] Setup')

In [ ]:
# OPTIONS
ENABLE_NARRATION = True
ENABLE_MUSIC = plan.get('enable_music', False)
ENABLE_SUBTITLES = plan.get('enable_subtitles', False)
ENABLE_CREDITS = plan.get('enable_credits', True)
MUSIC_PROMPT = 'Ambient cinematic background music'
MUSIC_VOLUME = 0.10
music_path = None; credits_path = None
print('[SCENE] Options:')
for n,v in [('Narration',ENABLE_NARRATION),('Music',ENABLE_MUSIC),('Subtitles',ENABLE_SUBTITLES),('Credits',ENABLE_CREDITS)]:
    print(f'   {n}: {"[OK]" if v else "[!!]"}')

In [ ]:
# HELPERS
def dur(p):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(p)],capture_output=True,text=True,timeout=10).stdout.strip())
    except: return 5.0
def ff(cmd,desc=''):
    if desc: print(f'   {desc}...',end=' ',flush=True)
    r=subprocess.run(cmd,capture_output=True,text=True,timeout=600)
    if r.returncode==0:
        if desc: print('[OK]')
        return True
    if desc: print('[!!]')
    print(f'   {r.stderr[-300:]}'); return False
def build_effect_filter(name, intensity=0.7):
    i=max(0.0,min(1.0,intensity))
    if name=='film_grain': return f'noise=alls={int(12*i)}:allf=t'
    if name=='vhs_artifacts': return f'noise=alls={int(30*i)}:allf=t,rgbashift=rh={int(2*i)}:bh={int(-2*i)},eq=contrast={1+0.1*i:.3f}:brightness={0.02*i:.3f}:saturation={1-0.15*i:.3f}'
    if name=='celluloid_scratches': return f'noise=c0s={int(15*i)}:c0f=t:c1s={int(15*i)}:c1f=t,eq=contrast={1+0.05*i:.3f}:brightness={-0.01*i:.3f}'
    if name=='sepia_tone': inv=1-i; return f'colorchannelmixer={inv+i*0.393:.3f}:{i*0.769:.3f}:{i*0.189:.3f}:0:{i*0.349:.3f}:{inv+i*0.686:.3f}:{i*0.168:.3f}:0:{i*0.272:.3f}:{i*0.534:.3f}:{inv+i*0.131:.3f}:0'
    if name=='bw_transition': inv=1-i; return f'colorchannelmixer={inv+i*0.3:.3f}:{i*0.4:.3f}:{i*0.3:.3f}:0:{i*0.3:.3f}:{inv+i*0.4:.3f}:{i*0.3:.3f}:0:{i*0.3:.3f}:{i*0.4:.3f}:{inv+i*0.3:.3f}:0'
    if name=='color_grade_warm': return f'eq=saturation={1+0.1*i:.3f}:brightness={0.02*i:.3f},colorbalance=rs={0.1*i:.3f}:gs={0.05*i:.3f}:bs={-0.05*i:.3f}'
    if name=='color_grade_cool': return f'eq=saturation={1-0.1*i:.3f},colorbalance=rs={-0.05*i:.3f}:gs=0:bs={0.12*i:.3f}'
    if name=='vignette': return f'vignette=PI/4*{i:.3f}'
    if name=='flicker': return f"noise=alls={int(8*i)}:allf=t,eq=brightness='{0.02*i:.4f}*sin(2*PI*t*3)'"
    if name=='desaturate': return f'eq=saturation={0.5+0.5*(1-i):.3f}'
    return None

In [ ]:
# STEP 1: BUILD SEQUENCE FROM STORYBOARD
print(f'[TOOL] Step 1: Building {len(storyboard)} scenes from storyboard...')
norm=[]; arc_idx=0; upl_idx=0
for sc in storyboard:
    sn=sc['scene'];stype=sc['scene_type']
    icon='[AI]' if stype=='ai_generated' else '[ARCH]' if stype=='archive' else '[DIR]'
    out=TEMP/f'scene_{sn:03d}.mp4'; src=None
    if stype=='ai_generated':
        c=VIDEOS_DIR/f'scene_{sn:03d}.mp4'
        if c.exists(): src=c
        else: print(f'   {icon} Scene {sn}: [!!] missing (run 07)'); continue
    elif stype=='archive':
        if arc_idx<len(archive_clips): src=Path(archive_clips[arc_idx]['path']); arc_idx+=1
        if not src or not src.exists(): print(f'   {icon} Scene {sn}: [!!] missing (run 03)'); continue
    elif stype=='upload':
        if upl_idx<len(upload_clips): src=Path(upload_clips[upl_idx]['path']); upl_idx+=1
        if not src or not src.exists(): print(f'   {icon} Scene {sn}: [!!] missing (run 04)'); continue
    vf='scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2:black'
    eff_tag=''
    if stype=='ai_generated' and sn in effects_map:
        p=effects_map[sn]
        for eff in p.get('ffmpeg_effects',[]):
            f=build_effect_filter(eff,p.get('effect_intensity',0.7))
            if f: vf+=','+f
        eff_tag=f' +{p.get("ffmpeg_effects",[])}'
    if ff(['ffmpeg','-y','-i',str(src),'-vf',vf,'-r','24','-pix_fmt','yuv420p','-c:v','libx264','-preset','fast','-crf','23','-an','-t',str(sc.get('duration',5)),str(out)],f'{icon} Scene {sn}{eff_tag}'):
        norm.append(out)
print(f'[OK] {len(norm)}/{len(storyboard)} scenes')

In [ ]:
# STEP 2: MUSIC
if ENABLE_MUSIC:
    print('[MUSIC] Step 2: Music...')
    try:
        import fal_client, time, requests
        if not os.environ.get('FAL_KEY'): raise Exception('No FAL_KEY')
        td=sum(dur(v) for v in norm)+10
        result=fal_client.subscribe('beatoven/music-generation',arguments={'prompt':MUSIC_PROMPT,'duration':min(150,int(td))})
        url=result.get('audio',{}).get('url') if isinstance(result.get('audio'),dict) else result.get('audio') or result.get('url','')
        if url: music_path=TEMP/'music.wav'; open(music_path,'wb').write(requests.get(url,timeout=120).content); print('   [OK]')
    except Exception as e: print(f'   [!!] {e}')
else: print('[MUSIC] Step 2: skipped')

In [ ]:
# STEP 3: CREDITS
if ENABLE_CREDITS:
    print('[SCROLL] Step 3: Credits...')
    title=essay.get('title','Untitled') if essay else 'Untitled'
    sources=essay.get('credits',{}).get('sources',[]) if essay else []
    lines=[title,'','Generated by ROTBOTS','','-- Sources --']+[s[:60] for s in sources]+['','-- Pipeline --','LLM: Groq','Video: fal.ai','Voice: Edge-TTS','FFmpeg','','LI-MA TDA 2026']
    D=8;LH=42;spd=(720+len(lines)*LH)/D
    flt=[f"drawtext=text='{l.replace(chr(39),chr(8217)).replace(chr(58),chr(92)+chr(58))}':fontsize=28:fontcolor=white:x=(w-text_w)/2:y=h+{i*LH}-{spd:.1f}*t" for i,l in enumerate(lines) if l]
    credits_path=TEMP/'credits.mp4'
    ff(['ffmpeg','-y','-f','lavfi','-i',f'color=c=black:s=1280x720:d={D}:r=24','-vf',','.join(flt),'-pix_fmt','yuv420p','-c:v','libx264','-preset','fast',str(credits_path)],'Credits')
else: print('[SCROLL] Step 3: skipped')

In [ ]:
# STEPS 4-6: CONCAT + AUDIO (with padding) + SUBTITLES
print('[SCENE] Step 4: Concat...')
cf=TEMP/'concat.txt'
with open(cf,'w') as f:
    for v in norm: f.write(f"file '{v}'\n")
    if credits_path and credits_path.exists(): f.write(f"file '{credits_path}'\n")
concat_out=TEMP/'concatenated.mp4'
ff(['ffmpeg','-y','-f','concat','-safe','0','-i',str(cf),'-c','copy',str(concat_out)],'Concat')
video_duration = dur(concat_out)
print(f'   {video_duration:.1f}s')

audio_out=TEMP/'with_audio.mp4'
has_narr=ENABLE_NARRATION and has_narration_file
has_mus=music_path is not None and music_path.exists()
if has_narr:
    print('[MIC] Step 5: Audio...')
    # Concat all narration files
    acf=TEMP/'ac.txt'
    with open(acf,'w') as f:
        for a in audio_files: f.write(f"file '{a}'\n")
    narr=TEMP/'narration.mp3'
    ff(['ffmpeg','-y','-f','concat','-safe','0','-i',str(acf),'-c','copy',str(narr)],'Concat narration')
    
    # Pad narration with silence to match video length (fixes credits being cut off)
    narr_padded=TEMP/'narration_padded.mp3'
    ff(['ffmpeg','-y','-i',str(narr),'-af',f'apad=whole_dur={video_duration}','-c:a','libmp3lame','-b:a','128k',str(narr_padded)],'Pad audio to video length')
    
    # Mix with music if enabled
    audio_track = narr_padded
    if has_mus:
        mx=TEMP/'mixed.mp3'
        ff(['ffmpeg','-y','-i',str(narr_padded),'-i',str(music_path),'-filter_complex',f'[0:a]volume=1.0[n];[1:a]volume={MUSIC_VOLUME}[m];[n][m]amix=inputs=2:duration=first[out]','-map','[out]','-c:a','libmp3lame',str(mx)],'Mix narration + music')
        audio_track = mx
    
    # Merge audio with video (no -shortest needed since audio is already padded)
    ff(['ffmpeg','-y','-i',str(concat_out),'-i',str(audio_track),'-c:v','copy','-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0',str(audio_out)],'Merge audio + video')
elif has_mus:
    ff(['ffmpeg','-y','-i',str(concat_out),'-i',str(music_path),'-c:v','copy','-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0','-shortest',str(audio_out)],'Music only')
else: shutil.copy2(str(concat_out),str(audio_out))

# Subtitles
final=SESSION_DIR/'final_video.mp4'
if ENABLE_SUBTITLES and sub_file.exists():
    print('[NOTE] Step 6: Subtitles...')
    la=TEMP/'subtitles.ass';shutil.copy2(str(sub_file),str(la))
    if not ff(['ffmpeg','-y','-i',str(audio_out),'-vf',f'ass={str(la)}','-c:v','libx264','-preset','fast','-crf','23','-c:a','copy',str(final)],'Burn subs (ass)'):
        if not ff(['ffmpeg','-y','-i',str(audio_out),'-vf',f'subtitles={str(la)}','-c:v','libx264','-preset','fast','-crf','23','-c:a','copy',str(final)],'Burn subs (subtitles)'):
            shutil.copy2(str(audio_out),str(final))
else: shutil.copy2(str(audio_out),str(final))
if final.exists(): print(f'\n[OK] Final: {dur(final):.1f}s, {final.stat().st_size/(1024*1024):.1f}MB')

---
## [SCENE] Watch!

In [ ]:
title=essay.get('title','Untitled') if essay else 'Untitled'
if final.exists():
    display(Markdown(f'# [SCENE] {title}\n*Generated by ROTBOTS*\n\n---'))
    try: display(Video(str(final),embed=True,width=720))
    except: print(f'Drive: rotbots_workshop/{SESSION_NAME}/final_video.mp4')
else: print('[!!]')

In [ ]:
# SUMMARY
print('[?] Pipeline Summary')
print('='*50)
st=[]
st.append(f'[AI] {ai_count} AI scenes')
if arc_count: st.append(f'[ARCH] {arc_count} archive scenes')
if upl_count: st.append(f'[DIR] {upl_count} upload scenes')
if effects_map: st.append(f'[ART] {len(effects_map)} effects')
st.append(f'[MIC] {len(audio_files)} narrations')
if ENABLE_MUSIC and music_path: st.append('[MUSIC] Music')
if ENABLE_SUBTITLES and sub_file.exists(): st.append('[NOTE] Subtitles')
if ENABLE_CREDITS: st.append('[SCROLL] Credits')
if final.exists(): st.append(f'[FILM] {dur(final):.1f}s')
for s in st: print(f'   {s}')

---
**Every step: automated decisions.** What does that mean?

---
*ROTBOTS -- LI-MA TDA 2026, Amsterdam*


---

# [END] Pipeline Complete

Every step: automated decisions. What does that mean?

---
*ROTBOTS -- LI-MA TDA 2026, Amsterdam*